In [2]:
import pandas as pd
import numpy as np

In [3]:
GLOBALPARAMETER = pd.read_csv("ergebnisse/ergebnisseAnalyseGLOBAL.csv")
JAHRESPARAMETER = pd.read_csv("ergebnisse/ergebnisseAnalyseJAHRES.csv")

In [4]:
# KENNZAHLEN

In [5]:
def aggregateKPIs(df, groupCols):
    return (
        df.groupby(groupCols, observed = True)
        .agg(
            hitRate=("reaction", "mean"),
            meanMaxZ=("maxZ", "mean"),
            nEvents=("reaction", "size"),
        )
        .reset_index()
        .sort_values(groupCols)
    )

# HAUPTANALYSE

In [6]:
resultsMethod = aggregateKPIs(GLOBALPARAMETER, ["methodGroup"])
resultsMethodMarketphase = aggregateKPIs(GLOBALPARAMETER, ["methodGroup", "marketphase"])
resultsMethodAssetType = aggregateKPIs(GLOBALPARAMETER, ["methodGroup", "assetType"])
resultsMethodAsset = aggregateKPIs(GLOBALPARAMETER, ["methodGroup", "asset"])

In [7]:
# GESAMTEBENE

In [8]:
resultsMethod

,methodGroup,hitRate,meanMaxZ,nEvents
0,fibonacci,0.508261,2.432136,3571
1,support_resistance,0.510562,2.400218,4734


In [9]:
# MARKTPHASE

In [10]:
resultsMethodMarketphase

,methodGroup,marketphase,hitRate,meanMaxZ,nEvents
0,fibonacci,sideways,0.499000,2.414291,2499
1,fibonacci,trend,0.529851,2.473737,1072
2,support_resistance,sideways,0.510645,2.417356,3006
3,support_resistance,trend,0.510417,2.370404,1728


In [11]:
# FINANZINSTRUMENT

In [12]:
resultsMethodAssetType

,methodGroup,assetType,hitRate,meanMaxZ,nEvents
0,fibonacci,MarketIndex,0.506969,2.428349,1148
1,fibonacci,Stock,0.508873,2.433931,2423
2,support_resistance,MarketIndex,0.503097,2.403595,1453
3,support_resistance,Stock,0.513868,2.398722,3281


# ZUSATZANALYSE

In [13]:
# Level Value

In [14]:
levelValues = aggregateKPIs(GLOBALPARAMETER, ["levelLabel"])
levelValues.sort_values(by=["levelLabel", "hitRate"])

,levelLabel,hitRate,meanMaxZ,nEvents
0,fib_0.236,0.614740,2.778200,597
1,fib_0.382,0.527297,2.587532,751
2,fib_0.5,0.515306,2.434560,784
3,fib_0.618,0.464194,2.293292,782
4,fib_0.786,0.433790,2.102417,657
5,resistance_1,0.516903,2.440596,917
6,resistance_2,0.467914,2.283601,748
7,resistance_3,0.452880,2.207024,382
8,resistance_4,0.460317,2.185301,126
9,resistance_5,0.600000,2.238697,25


In [15]:
# Eventanzahl

In [16]:
eventCounts = GLOBALPARAMETER.groupby(["asset", "year", "methodGroup", "levelLabel"]).agg(levelCount = ("levelLabel","size")).reset_index()
eventCounts["levelGroupCount"] = pd.cut(eventCounts["levelCount"], bins=[0, 2, 4, 6, 8, np.inf],labels=["1-2", "3-4", "5-6", "7-8", "9+"])
GLOBALTOUCHcount  = GLOBALPARAMETER.merge(eventCounts, on=["asset", "year", "methodGroup", "levelLabel"], how="left")
aggregateKPIs(GLOBALTOUCHcount , ["methodGroup", "levelGroupCount"])

,methodGroup,levelGroupCount,hitRate,meanMaxZ,nEvents
0,fibonacci,1-2,0.604790,2.987308,167
1,fibonacci,3-4,0.549550,2.696934,555
2,fibonacci,5-6,0.489627,2.384536,723
3,fibonacci,7-8,0.474149,2.263766,793
4,fibonacci,9+,0.509377,2.378316,1333
5,support_resistance,1-2,0.482143,2.219469,168
6,support_resistance,3-4,0.494302,2.362845,702
7,support_resistance,5-6,0.527453,2.441954,1111
8,support_resistance,7-8,0.522078,2.491350,1155
9,support_resistance,9+,0.500626,2.340753,1598


In [17]:
# Überlappung

In [18]:
levelDF = GLOBALPARAMETER[["asset", "year", "methodGroup", "levelLabel", "levelValue"]].drop_duplicates()
fibLEVELS = levelDF[levelDF["methodGroup"] == "fibonacci"].copy()
srLEVELS = levelDF[levelDF["methodGroup"] == "support_resistance"].copy()
overlapLEVELS = fibLEVELS.merge(srLEVELS, on=["asset", "year"], suffixes=("_fib", "_sr"))

overlapLEVELS["distance"] = abs(overlapLEVELS["levelValue_fib"] - overlapLEVELS["levelValue_sr"]) / overlapLEVELS["levelValue_fib"]
overlapLEVELS["overlapType"] = np.where(overlapLEVELS["distance"] <= 0.01, "overlapping", "isolated") # 0.01

onlyOverlaps = overlapLEVELS[overlapLEVELS["overlapType"] == "overlapping"]

fibOverlap = (
    onlyOverlaps[["asset", "year", "methodGroup_fib", "levelLabel_fib"]]
    .drop_duplicates()
    .rename(columns={
        "methodGroup_fib": "methodGroup",
        "levelLabel_fib": "levelLabel"
    })
)

srOverlap = (
    onlyOverlaps[["asset", "year", "methodGroup_sr", "levelLabel_sr"]]
    .drop_duplicates()
    .rename(columns={
        "methodGroup_sr": "methodGroup",
        "levelLabel_sr": "levelLabel"
    })
)

overlapInfo = pd.concat([fibOverlap, srOverlap], ignore_index=True).drop_duplicates()
overlapInfo["overlapType"] = "overlapping"

GLOBALOVERLAP = GLOBALPARAMETER.merge(
    overlapInfo,
    on=["asset", "year", "methodGroup", "levelLabel"],
    how="left"
)

GLOBALOVERLAP["overlapType"] = GLOBALOVERLAP["overlapType"].fillna("isolated")

aggregateKPIs(GLOBALOVERLAP, ["methodGroup", "overlapType"])

,methodGroup,overlapType,hitRate,meanMaxZ,nEvents
0,fibonacci,isolated,0.536671,2.615566,1718
1,fibonacci,overlapping,0.481921,2.262070,1853
2,support_resistance,isolated,0.535122,2.500886,2534
3,support_resistance,overlapping,0.482273,2.284266,2200


# PARAMETRISIERUNG - Robustheitprüfung

In [19]:
print(f"GLOBAL: {len(GLOBALPARAMETER)}")
print(f"JAHR: {len(JAHRESPARAMETER)}")

GLOBALresultsMethod = aggregateKPIs(GLOBALPARAMETER, ["methodGroup"])
YEARresultsMethod = aggregateKPIs(JAHRESPARAMETER, ["methodGroup"])

globalPlot = GLOBALresultsMethod.copy()
globalPlot["parametrisierung"] = "Global"

yearPlot = YEARresultsMethod.copy()
yearPlot["parametrisierung"] = "Jährlich"

paramCompare = pd.concat([globalPlot, yearPlot], ignore_index=True)
paramCompare

GLOBAL: 8305
JAHR: 8367


,methodGroup,hitRate,meanMaxZ,nEvents,parametrisierung
0,fibonacci,0.508261,2.432136,3571,Global
1,support_resistance,0.510562,2.400218,4734,Global
2,fibonacci,0.552339,2.521421,3592,Jährlich
3,support_resistance,0.545759,2.473418,4775,Jährlich
